# Find and Replace
This notebook is useful for correcting small errors in a collection without having to reingest. I created it to replace the "source_key" of "paper" to "lsst_bib". It works fine, but it occasionally stops running for reasons I haven't been able to figure out. If you just start it up again, it should keep running. I used this in tandem with the `count_property` function in `view_collection.ipynb` to track progress. 

In [ ]:
import os
import logging

from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.config import AdditionalConfig, Timeout
from weaviate import WeaviateClient
from weaviate.classes.query import Filter

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

In [ ]:
def find_and_replace(client: WeaviateClient,
                     index_name: str,
                     prop: str,
                     old_key: str,
                     new_key: str,
) -> None:
    """Find all instances of a given property and replace the key
    with a new value.
    
    Parameters
    ----------
    client: WeaviateClient
        The connection to the Weaviate Client.
    index_name: str
        Name of the collection to edit.
    prop: str
        The metadata property to change (e.g. source_key, source)
    old_key: str
        The name of the old key to filter by
    new_key: str
        The new key to replace for the given property
    """
    collection = client.collections.get(index_name)
    limit = 100
    offset = 0
    
    while True:
        results = collection.query.fetch_objects(
            filters=Filter.by_property(prop).equal(old_key),
            limit=limit,
            offset=offset
        )
        if not results.objects:
            break
    
        for obj in results.objects:
            uuid = obj.uuid
            properties = obj.properties
            properties[prop] = new_key
            collection.data.update(
                uuid=uuid,
                properties=properties
            )
    
        offset += limit
        if len(results.objects) < limit:
            break

In [ ]:
try:
    client = weaviate.connect_to_custom(
        http_host=http_host,
        http_port=8080,  # Default is 80, WCD uses 443
        http_secure=False,
        grpc_host=grpc_host,
        grpc_port=50051,  # Default is 50051, WCD uses 443
        grpc_secure=False,
        auth_credentials=Auth.api_key(
            weaviate_api_key
        ),  # The API key to use for authentication
        headers={"X-OpenAI-Api-Key": openai_api_key},
        additional_config=AdditionalConfig(
            timeout=Timeout(init=30, query=60, insert=300)
        )
    )
    print("Client is live:", client.is_ready())

    INDEX_NAME = "Ingestion_20250610"

    find_and_replace(client=client,
                     index_name=INDEX_NAME,
                     prop="source_key",
                     old_key="paper",
                     new_key="lsst_bib")
    
except Exception as e:
    print(e)
finally:
    client.close()